In [ ]:
import numpy as np
import pandas as pd
import kaggle

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
# Load csv from file location, if the data was downloaded using the program provided then leave the location as is
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv", low_memory=False)

In [ ]:
# Column names

print(df.columns.to_series().to_string())

In [ ]:
# Get rid of na dates

df = df[df["issue_d"].notna()]

In [ ]:
# Check all na dates have been removed
df["issue_d"].isna().sum()

In [ ]:
# Filter dates between 2012 and 2017

df["issue_d"] = pd.to_datetime(df["issue_d"])
df1 = df[(df["issue_d"].dt.year >= 2012) & (df["issue_d"].dt.year <= 2017)]

# Filter to rows where loan outcome is known 

df1 = df1[df1["loan_status"].isin(["Default", "Charged Off", "Fully Paid"])]

In [ ]:
df1["loan_status"].unique()

In [ ]:
# Change target variable to binary

df1["target"] = df1["loan_status"].isin(["Charged Off", "Default"]).astype(int)

In [ ]:
# Check date filtering

date_head = df1.sort_values(by ="issue_d").head()
date_tail = df1.sort_values(by ="issue_d").tail()

display(date_head)
display(date_tail)

In [ ]:
# Define feature categories

credit_profile = ["fico_range_low", "fico_range_high", "inq_last_6mths", "delinq_2yrs", "open_acc", "total_acc", "pub_rec",
                  "pub_rec_bankruptcies", "revol_util", "revol_bal", "mort_acc", "acc_now_delinq"]

loan_characteristics = ["loan_amnt", "term", "int_rate", "installment", "sub_grade", "purpose", "application_type",
                       "initial_list_status"]

employment_and_income = ["annual_inc", "emp_length", "verification_status", "dti"]

borrower_stability = ["home_ownership", "addr_state"]

timing = ["issue_d", "earliest_cr_line"]

target = ["loan_status", "target"]

model_features = (credit_profile + loan_characteristics + employment_and_income + borrower_stability + timing + target)

model_features

In [ ]:
# Create a dataframe with only the feature variables
df_feat = df1[model_features]

In [ ]:
# Create a column to measure credit history length then drop issue date and earliest credit line
df_feat["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"])
df_feat["cr_length_hist"] = ((df_feat["issue_d"] - df_feat["earliest_cr_line"]).dt.days)
df_feat["cr_length_hist"] = df_feat["cr_length_hist"]/365.25
df_feat["cr_length_hist"]

# Drop issue date and earliest credit line column

df_feat = df_feat.drop(columns = ["earliest_cr_line", "issue_d"])

In [ ]:
df_feat.head()

In [ ]:
df_feat.isna().sum()

In [ ]:
# Make sure all columns are suitable for encoding particularly dates
df_feat["term"] = df_feat["term"].str[1:3]

# Sort NA rows

# Drop the 1 na row
df_feat = df_feat[df_feat["inq_last_6mths"].notna()]

# NA in both revol util and mort acc means no history therefore fill with 0
df_feat["revol_util"] = df_feat["revol_util"].fillna(0)
df_feat["mort_acc"] = df_feat["mort_acc"].fillna(0)

# DTI only has 240 missing rows so drop them as its a small proportion
df_feat = df_feat[df_feat["dti"].notna()]

# For missing employment length fill with Unknown
df_feat["emp_length"] = df_feat["emp_length"].fillna("Unknown")


In [ ]:
# Check all nan have been changed
df_feat.isna().sum()

In [ ]:
df_feat.head()

In [ ]:
# Before encoding check and change data types

df_feat.dtypes

In [ ]:
# Change term to integer and also change application_type and initial_list_status to binary encoding as they have 2 unique values
df_feat["term"] = df_feat["term"].astype(int)
df_feat["application_type"] = (df_feat["application_type"] == "Individual").astype(int)
df_feat["initial_list_status"] = (df_feat["initial_list_status"] == "w").astype(int)

df_feat.dtypes

In [ ]:
df_feat["emp_length"].unique()

In [ ]:
# Begin Encoding each column that isnt a float or integer

# Ordinal mapping for sub-grade and employment length

employment_len_map = {"Unknown": -1, "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5,
                         "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10}

grades = ["A", "B", "C", "D", "E", "F", "G"]
sub_grade_map = {f"{g}{n}": i*5 + n for i, g in enumerate(grades) for n in range(1, 6)}

# Apply ordinal maps
df_feat["emp_length"] = df_feat["emp_length"].map(employment_len_map)
df_feat["sub_grade"] = df_feat["sub_grade"].map(sub_grade_map)

In [ ]:
# Apply one-hot encoding for remaining object columns apart from addr_state

df_feat = pd.get_dummies(df_feat, columns = ["purpose", "application_type", "verification_status", "home_ownership"], drop_first = True)

In [ ]:
df_feat.head()

In [ ]:
    df_feat_final = df_feat.drop(columns = ["loan_status", "target"])
df_feat_final.head()

In [ ]:
df_feat_final.isna().sum()

In [ ]:
df_feat_final.dtypes